# Lab 78: The Sugar Pill That Heals

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 9, *The Future Interface*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-78.ipynb)

**What you will do:** time how long it takes you to feel relief after a painkiller (or run the blind coffee test on a willing friend), compare your own latency with how fast the drug could plausibly be working, and then run a small simulation of how a biased "AI flag" can nudge a decision-maker's judgement the same way a belief nudges the body.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

This Lab involves tasting (coffee, and optionally wine) and briefly deceiving a willing participant about
what they are drinking. Only run the coffee or wine steps with an adult who has agreed in advance to be
deceived about which drink is which, and tell them the truth immediately afterwards. If you use wine,
never pressure anyone to drink alcohol, offer a non-alcoholic substitute for anyone who prefers it, and
make sure no one drives after drinking. The coffee/decaf version needs no alcohol and works on its own.

## The experiment in the book

In Lab 78 you learned that the **placebo response** is a genuine physiological effect: believing a
treatment will work can trigger real neurochemical changes (endogenous opioids, dopamine release) that
produce real relief, even when the treatment itself is inert. You tried tracking your own painkiller
timing, and optionally ran the blind coffee or wine-label experiments on a friend.

## Record your results

Track your own placebo-relevant timing: each time you take a painkiller (or a caffeinated drink), note
the time you took it and the time you first noticed a change. Most oral painkillers need 15-30 minutes
to reach the bloodstream and take effect (as the Lab's own reflection step states), so relief reported
faster than that cannot be the drug itself.

In [ ]:
import pandas as pd

# example data: replace with your own timings, in minutes after taking the painkiller/drink
doses = pd.DataFrame([
    {"event": "painkiller, headache",  "minutes_to_relief": 3},
    {"event": "painkiller, back pain", "minutes_to_relief": 6},
    {"event": "coffee, tiredness",     "minutes_to_relief": 4},
])  # example data: replace with your own
display(doses)
doses["minutes_to_relief"].plot.barh(color="steelblue", figsize=(6, 2.2))
plt.xlabel("minutes to first noticed relief")
plt.axvline(15, color="firebrick", linestyle="--", label="fastest plausible drug action (15 min)")
plt.legend(); plt.gca().invert_yaxis(); plt.show()

## Compare

The Lab states that "most oral painkillers take 15-30 minutes to take effect", so relief felt faster
than that "cannot be the drug working" and must be, at least in part, a placebo response.

In [ ]:
fastest_plausible = 15
placebo_like = doses[doses["minutes_to_relief"] < fastest_plausible]
print(f"{len(placebo_like)} of {len(doses)} logged events reported relief before the drug could plausibly act.")
if len(placebo_like):
    display(placebo_like)
    print("These are candidates for a placebo response, in the Lab's own sense of the term.")
else:
    print("None of your logged events were faster than 15 minutes -- either you didn't experience a fast")
    print("placebo response this time, or you weren't attending closely to the exact time of relief.")

## The AI side

The Lab's AI Connection describes a technologically mediated version of the same effect: when a doctor
is told an AI system has "flagged" a scan as abnormal, the doctor becomes more likely to report finding
an abnormality -- even when the flag is a false positive. The flag changes the human's expectation, and
the expectation changes what the human reports. Below, a toy simulation shows the same mechanism in
miniature: a population of simulated "readers" judging identical, genuinely ambiguous scans, some of
them shown a false AI flag before they judge.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n_readers = 400

# Each simulated scan has the same, genuinely ambiguous evidence: a noisy signal centred at 0 (truly
# normal), with no reliable abnormality present in this batch.
evidence = rng.normal(0, 1, n_readers)

# Without any AI flag, a reader calls "abnormal" if their perceived evidence exceeds a fixed threshold.
threshold = 1.0
no_flag_calls = evidence > threshold

# With a false "AI flag" shown before judgement, the flag nudges perceived evidence upward -- the
# expectation effect the book describes -- so the same physical evidence more often crosses the
# threshold.
expectation_boost = 0.9
flagged_calls = (evidence + expectation_boost) > threshold

results = pd.DataFrame({
    "condition": ["no AI flag shown", "false AI flag shown"],
    "false abnormal calls": [no_flag_calls.mean(), flagged_calls.mean()],
})
display(results)
results.set_index("condition")["false abnormal calls"].plot.bar(color=["grey", "firebrick"], figsize=(5, 3))
plt.ylabel("proportion calling a normal scan 'abnormal'"); plt.xticks(rotation=15); plt.show()
print(f"The false flag alone raised the false-alarm rate from {no_flag_calls.mean():.0%} to {flagged_calls.mean():.0%},")
print("with the underlying evidence identical in both conditions -- the same logic as the placebo effect,")
print("moved from the reader's body to the reader's judgement.")

This is a toy model, not a study of real radiologists: the expectation_boost value was chosen to make
the effect visible, not measured from data. Set it to 0 and rerun to confirm the effect disappears when
there is nothing to bias the reader. The deeper limit of the analogy: your own placebo response involves
real neurochemistry (endogenous opioids, dopamine) changing what your body actually does, whereas this
simulated "reader" is a single threshold rule with no body and no belief -- it is a model of the *bias*,
not a model of a mind that holds an expectation.

## Questions to think about

1. In the simulation, both conditions see the exact same distribution of underlying evidence. What does that control for, and why is it essential to isolating the effect of the flag itself?
2. Set expectation_boost to a small number (for example, 0.1) instead of 0.9. Does the false-alarm rate still rise, and if so, by how much less? What does this suggest about whether the size of an expectation effect is fixed or graded?
3. The book's open-label placebo section reports that patients told, honestly, that their pills were inert still improved more than patients given no pills at all. Does the simulation above have any equivalent of an "honest" flag -- one that changes behaviour through ritual rather than through a false belief? If not, what would you need to add to model that?
4. If you ran the coffee or wine-label experiment on a friend, did their reported experience change in the direction the label suggested? Compare their result with your own timing data above -- do you see any parallel between the body's placebo response and a judgement-level expectation effect?

## Challenge

With a willing friend, run the coffee experiment from Step 2 (two identical decaffeinated coffees, one
described as caffeinated) and rate their reported alertness on a 1-10 scale immediately, and again after
20 minutes. Compare the two ratings.

In [ ]:
# Example data: replace with your friend's own ratings, 1 (not at all alert) to 10 (very alert).
rating_immediate = 5  # example data: rated right after drinking, before any real caffeine could act
rating_20min = 7      # example data: rated 20 minutes later

print(f"Change in reported alertness: {rating_20min - rating_immediate:+d} points")
print("Both cups were decaffeinated. Any rise here is consistent with a placebo response, though a single")
print("friend on a single occasion cannot rule out them simply feeling more awake as time passed anyway --")
print("a genuine control condition (an honestly-labelled decaf cup, on a different day) would rule that out.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-78.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")